# Stage 5 — IDM-VTON Pre-trained Inference

**What we built so far:**
- Stage 1: Pose skeleton (MediaPipe Tasks API)
- Stage 2: Body segmentation → cloth-agnostic person + erase mask (SegFormer)
- Stage 3: Clothing placement from erase mask bounding box
- Stage 4: Poisson blending + histogram matching → seamless composite

**Stage 4's limitation:** Poisson blending is a classical method. It desaturates colour and can't reconstruct realistic fabric wrinkles or lighting interactions.

**Stage 5 upgrade:** Run [IDM-VTON](https://github.com/yisol/IDM-VTON) — a state-of-the-art **diffusion model** trained specifically for virtual try-on. It replaces our Poisson blend with learned, photorealistic synthesis.

---

### How IDM-VTON works

IDM-VTON extends **Stable Diffusion XL (SDXL)** inpainting with two innovations:

1. **Garment Net** — a second UNet that encodes the garment image. Its features are injected into the main UNet via **IP-Adapter cross-attention**, so the model "sees" fine garment details (texture, print, collar shape) during denoising.
2. **Try-on Net** — the main SDXL inpainting UNet conditioned on:
   - Cloth-agnostic person image (shirt erased)
   - Binary mask of the clothing region
   - Pose skeleton image
   - Garment features from the Garment Net

Our Stage 1-4 outputs map **directly** to these inputs:

| IDM-VTON input | Our output |
|---|---|
| Cloth-agnostic person | `agnostic_person.png` (Stage 2) |
| Inpainting mask | `erase_mask.png` (Stage 2) |
| Pose skeleton | `pose_output.jpg` (Stage 1) |
| Garment image | `clothing.jpg` (our input) |

We skip IDM-VTON's internal preprocessing because we already did it in Stages 1-3.

> **IMPORTANT: Use a FRESH Colab runtime** — Runtime → Factory reset runtime before running. This avoids the broken system `huggingface_hub` that caused issues in Stages 1-4.

In [ ]:
# Sanity check: confirm we have a T4 GPU and enough VRAM
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Runtime > Change runtime type > T4 GPU")

gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU   : {gpu_name}")
print(f"VRAM  : {total_vram:.1f} GB")
print()
if total_vram < 14:
    print("WARNING: IDM-VTON (SDXL) needs ~12 GB. Less than 14 GB may OOM.")
    print("         If you hit OOM, re-run the pipeline cell with cpu_offload=True")
else:
    print("VRAM looks sufficient. Memory tricks (xformers + attention slicing) will be enabled.")

## 1. Install Dependencies

### Two-step flow — read before running

**Step 1 →** Run the install cell below. It installs packages, then **automatically kills the runtime** (`os.kill`). Colab will show a *"Your session crashed"* banner — that is **expected and intentional**.

**Step 2 →** After Colab reconnects, **skip the install cell** and continue from the GPU check cell downward. The packages are already on disk; the restart just flushes the stale in-memory `.so` files so Python picks up the freshly installed versions.

---

**Why no `numpy<2.0` pin here:**
Colab Python 3.12 ships **numpy 2.x** and its C extensions (`mtrand.so`, etc.) are compiled against that ABI. Downgrading to numpy 1.x at runtime — without a restart — causes the binary-incompatibility crash you just saw:
```
ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 … got 88
```
Stage 5 doesn't use scipy at all, so the `<2.0` requirement from Stages 1-4 simply doesn't apply here. We let Colab keep its native numpy 2.x.

**xformers** provides memory-efficient attention — critical for fitting two SDXL UNets on a T4.

In [ ]:
# ── INSTALL CELL — run once, then skip after restart ──────────────────────────
# After this cell finishes it kills the Python process so the new packages load
# cleanly. Colab will show "session crashed" — reconnect and continue from the
# next cell (GPU check). Do NOT re-run this cell after restart.

!pip install -q huggingface_hub==0.22.2          # pin before diffusers pulls its own version
!pip install -q diffusers==0.28.2 transformers==4.40.2 accelerate==0.30.0
!pip install -q xformers                          # memory-efficient attention for T4
!pip install -q kagglehub omegaconf einops safetensors
!pip install -q "Pillow>=9,<12" opencv-python
# NOTE: numpy is intentionally NOT pinned — Colab Python 3.12 uses numpy 2.x
#       natively; downgrading it breaks the ABI without a restart.

print("Packages installed. Restarting runtime now...")
import os, time
time.sleep(1)   # let the print flush
os.kill(os.getpid(), 9)  # force runtime restart

In [ ]:
# ── START HERE after the runtime restarts ─────────────────────────────────────
# Verify that freshly installed packages are active.
# If you see an ImportError here, re-run the install cell and wait for restart.
import importlib
for pkg in ["diffusers", "transformers", "accelerate", "huggingface_hub", "torch", "numpy"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg:20s} {mod.__version__}")

## 2. Mount Google Drive + Load Stage 1-4 Outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

DRIVE      = "/content/drive/MyDrive/VirtualTryOn"
INPUT_DIR  = f"{DRIVE}/input_images"
OUTPUT_DIR = f"{DRIVE}/output_images"
CKPT_DIR   = f"{DRIVE}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# Load all Stage 1-4 outputs
person_img   = Image.open(f"{INPUT_DIR}/test_person.jpg").convert("RGB")
cloth_img    = Image.open(f"{INPUT_DIR}/clothing.jpg").convert("RGB")
agnostic_img = Image.open(f"{OUTPUT_DIR}/agnostic_person.png").convert("RGB")
mask_img     = Image.open(f"{OUTPUT_DIR}/erase_mask.png").convert("L")
pose_img     = Image.open(f"{OUTPUT_DIR}/pose_output.jpg").convert("RGB")

images = {
    "person": person_img, "cloth": cloth_img,
    "agnostic": agnostic_img, "mask": mask_img, "pose": pose_img,
}
for name, img in images.items():
    print(f"  {name:12s} {img.size}  mode={img.mode}")

# Quick preview
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
    ax.set_title(name)
    ax.axis('off')
plt.suptitle("Stage 1-4 outputs loaded from Drive", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Download VITON-HD Dataset

VITON-HD is the standard benchmark for virtual try-on. It contains 1,024 test pairs with:
- `image/` — full person photos at 768×1024
- `cloth/` — product garment shots
- `agnostic-v3.2/` — pre-computed cloth-agnostic person (like our Stage 2 output)
- `agnostic-mask/` — pre-computed erase masks (like our Stage 2 output)
- `openpose-img/` — pre-computed DWPose skeletons (like our Stage 1 output)

We **won't save this to Drive** (it's ~10 GB). We download it fresh each session.

**Learning insight:** Compare the dataset's `agnostic-v3.2/` with our `agnostic_person.png` — they should look similar. This validates that Stages 1-3 are producing the right intermediate representations.

In [ ]:
# Set Kaggle token and download VITON-HD
os.environ['KAGGLE_TOKEN'] = 'KGAT_fbc21b5329e8cbe6a20395ed9e25c159'
import kagglehub

print("Downloading VITON-HD dataset (~10 GB, takes 5-10 min on Colab)...")
dataset_path = kagglehub.dataset_download("marquis03/high-resolution-viton-zalando-dataset")
print(f"\nDataset downloaded to: {dataset_path}")

In [ ]:
# Explore the dataset structure
print("Top-level folders:")
for item in sorted(os.listdir(dataset_path)):
    item_path = os.path.join(dataset_path, item)
    if os.path.isdir(item_path):
        print(f"  {item}/")
        for sub in sorted(os.listdir(item_path)):
            sub_path = os.path.join(item_path, sub)
            if os.path.isdir(sub_path):
                n = len(os.listdir(sub_path))
                print(f"    {sub}/  ({n} files)")

# Build paths to the test split
# Try common VITON-HD folder layouts
test_base = None
for candidate in ["test", "zalando-hd-resized/test", "viton-hd/test"]:
    p = os.path.join(dataset_path, candidate)
    if os.path.isdir(p):
        test_base = p
        break

if test_base is None:
    # Fallback: search for 'image' subfolder
    for root, dirs, files in os.walk(dataset_path):
        if 'image' in dirs and 'cloth' in dirs:
            test_base = root
            break

print(f"\nTest split root: {test_base}")
print("Subfolders:", sorted(os.listdir(test_base)))

In [ ]:
# Load a VITON-HD test pair to use as ground-truth comparison
img_dir   = os.path.join(test_base, "image")
cloth_dir = os.path.join(test_base, "cloth")

# Try to find the agnostic and mask folders (names vary slightly between dataset versions)
agn_dir  = None
mask_dir = None
pose_dir = None
for name in sorted(os.listdir(test_base)):
    low = name.lower()
    if 'agnostic' in low and 'mask' not in low and agn_dir is None:
        agn_dir = os.path.join(test_base, name)
    if 'agnostic' in low and 'mask' in low and mask_dir is None:
        mask_dir = os.path.join(test_base, name)
    if 'openpose' in low and 'img' in low and pose_dir is None:
        pose_dir = os.path.join(test_base, name)

print(f"image dir   : {img_dir}")
print(f"cloth dir   : {cloth_dir}")
print(f"agnostic dir: {agn_dir}")
print(f"mask dir    : {mask_dir}")
print(f"pose dir    : {pose_dir}")

# Pick the first test pair
test_img_files   = sorted(os.listdir(img_dir))
test_cloth_files = sorted(os.listdir(cloth_dir))
print(f"\nTest images available: {len(test_img_files)}")
print(f"Test cloths available: {len(test_cloth_files)}")
print(f"First person: {test_img_files[0]}")
print(f"First cloth : {test_cloth_files[0]}")

In [ ]:
# Load the first VITON-HD test pair
def find_file(directory, stem):
    """Find a file by stem (name without extension) in a directory."""
    if directory is None:
        return None
    for f in os.listdir(directory):
        if os.path.splitext(f)[0] == stem or f.startswith(stem):
            return os.path.join(directory, f)
    return None

vton_person_name = test_img_files[0]
vton_cloth_name  = test_cloth_files[0]
stem = os.path.splitext(vton_person_name)[0]

vton_person_path   = os.path.join(img_dir, vton_person_name)
vton_cloth_path    = os.path.join(cloth_dir, vton_cloth_name)
vton_agnostic_path = find_file(agn_dir, stem)  if agn_dir  else None
vton_mask_path     = find_file(mask_dir, stem) if mask_dir else None
vton_pose_path     = find_file(pose_dir, stem) if pose_dir else None

vton_person_img   = Image.open(vton_person_path).convert("RGB")
vton_cloth_img    = Image.open(vton_cloth_path).convert("RGB")
vton_agnostic_img = Image.open(vton_agnostic_path).convert("RGB") if vton_agnostic_path else None
vton_mask_img     = Image.open(vton_mask_path).convert("L")       if vton_mask_path     else None
vton_pose_img     = Image.open(vton_pose_path).convert("RGB")     if vton_pose_path     else None

print(f"Loaded VITON-HD test pair:")
print(f"  person  : {vton_person_img.size}")
print(f"  cloth   : {vton_cloth_img.size}")
print(f"  agnostic: {vton_agnostic_img.size if vton_agnostic_img else 'not found'}")
print(f"  mask    : {vton_mask_img.size     if vton_mask_img     else 'not found'}")
print(f"  pose    : {vton_pose_img.size     if vton_pose_img     else 'not found'}")

# Preview
avail = [(n, i) for n, i in [
    ("person", vton_person_img), ("cloth", vton_cloth_img),
    ("agnostic", vton_agnostic_img), ("mask", vton_mask_img), ("pose", vton_pose_img)
] if i is not None]
fig, axes = plt.subplots(1, len(avail), figsize=(4 * len(avail), 5))
if len(avail) == 1: axes = [axes]
for ax, (name, img) in zip(axes, avail):
    ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
    ax.set_title(f"VITON-HD: {name}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Set Up IDM-VTON

IDM-VTON uses a custom pipeline class (`StableDiffusionXLInpaintPipeline` subclass) and two custom `UNet2DConditionModel` subclasses. We clone their repo to get these, then load weights from HuggingFace.

**Model components loaded:**
- `unet` — the try-on UNet (SDXL, modified for garment cross-attention)
- `unet_encoder` — the garment UNet (encodes garment → features for IP-Adapter)
- `image_encoder` — CLIP vision encoder (for garment image embedding)
- `vae` — loaded from `madebyollin/sdxl-vae-fp16-fix` (numerically stable fp16 VAE)
- `scheduler` — DDIM scheduler

In [ ]:
%%capture
# Clone the IDM-VTON repo for the custom pipeline code
!git clone https://github.com/yisol/IDM-VTON /content/IDM-VTON
print("Cloned IDM-VTON.")

In [ ]:
# Add IDM-VTON source to Python path so we can import its custom classes
import sys
idmvton_root = "/content/IDM-VTON"
if idmvton_root not in sys.path:
    sys.path.insert(0, idmvton_root)

# Verify the custom modules are importable
from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
from src.unet_hacked_garmnet import UNet2DConditionModel as GarmentUNet
from src.unet_hacked_tryon   import UNet2DConditionModel as TryonUNet
print("IDM-VTON custom classes imported successfully.")

In [ ]:
# Download IDM-VTON weights from HuggingFace (~15 GB, takes 10-20 min)
# Note: NOT saved to Drive (too large). Re-download each session.
from huggingface_hub import snapshot_download

WEIGHTS_DIR = "/content/idm-vton-weights"
if not os.path.exists(os.path.join(WEIGHTS_DIR, "unet")):
    print("Downloading IDM-VTON weights (~15 GB)...")
    model_path = snapshot_download(
        repo_id="yisol/IDM-VTON",
        local_dir=WEIGHTS_DIR,
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],  # skip Flax variants
    )
else:
    model_path = WEIGHTS_DIR
    print("Weights already cached.")

print(f"Model path: {model_path}")
print("Subfolders:", sorted(os.listdir(model_path)))

In [ ]:
import torch
from diffusers import DDIMScheduler, AutoencoderKL
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection

DTYPE  = torch.float16
DEVICE = "cuda"

print("Loading model components in fp16...")

scheduler = DDIMScheduler.from_pretrained(model_path, subfolder="scheduler")
print("  scheduler loaded")

# fp16-stable VAE (avoids NaN during decode at fp16)
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", torch_dtype=DTYPE
).to(DEVICE)
print(f"  VAE loaded  ({torch.cuda.memory_allocated()/1e9:.1f} GB VRAM)")

unet = TryonUNet.from_pretrained(
    model_path, subfolder="unet", torch_dtype=DTYPE
).to(DEVICE)
print(f"  TryonUNet loaded  ({torch.cuda.memory_allocated()/1e9:.1f} GB VRAM)")

unet_encoder = GarmentUNet.from_pretrained(
    model_path, subfolder="unet_encoder", torch_dtype=DTYPE
).to(DEVICE)
print(f"  GarmentUNet loaded  ({torch.cuda.memory_allocated()/1e9:.1f} GB VRAM)")

image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    model_path, subfolder="image_encoder", torch_dtype=DTYPE
).to(DEVICE)
print(f"  CLIP image encoder loaded  ({torch.cuda.memory_allocated()/1e9:.1f} GB VRAM)")

feature_extractor = CLIPImageProcessor()

In [ ]:
# Assemble the pipeline
pipe = TryonPipeline.from_pretrained(
    model_path,
    unet=unet,
    vae=vae,
    feature_extractor=feature_extractor,
    text_encoder=None,    # IDM-VTON doesn't use SD text conditioning path
    text_encoder_2=None,
    tokenizer=None,
    tokenizer_2=None,
    scheduler=scheduler,
    image_encoder=image_encoder,
    torch_dtype=DTYPE,
)
pipe.unet_encoder = unet_encoder  # attach garment UNet
pipe.to(DEVICE)

# Memory optimizations — critical for T4 (15 GB VRAM with two SDXL UNets)
pipe.enable_attention_slicing()   # break attention into chunks to reduce peak VRAM
try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xformers memory-efficient attention enabled")
except Exception:
    print("xformers not available — attention slicing only")

torch.cuda.empty_cache()
print(f"\nPipeline ready.  VRAM allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 5. Prepare Inputs

IDM-VTON is trained on VITON-HD at **768 × 1024** (portrait). All person-derived images (person, agnostic, mask, pose) must be:
1. Scaled to the same size using the **same** transform — spatial alignment is critical
2. Resized to exactly 768 × 1024

Our `test_person.jpg` is **landscape** (1280 × 561), so we center-crop after scaling. The same crop is applied to `agnostic_person.png`, `erase_mask.png`, and `pose_output.jpg` to keep them aligned.

The garment image is processed independently (it's a product shot, not spatially tied to the person).

In [ ]:
TARGET_W, TARGET_H = 768, 1024

def compute_transform(img: Image.Image, target_w=TARGET_W, target_h=TARGET_H):
    """Compute scale + center-crop box so image fills target exactly."""
    w, h = img.size
    scale = max(target_w / w, target_h / h)
    new_w, new_h = round(w * scale), round(h * scale)
    left = (new_w - target_w) // 2
    top  = (new_h - target_h) // 2
    crop = (left, top, left + target_w, top + target_h)
    return scale, crop

def apply_transform(img: Image.Image, scale, crop, mode="RGB") -> Image.Image:
    img = img.convert(mode)
    w, h = img.size
    img = img.resize((round(w * scale), round(h * scale)), Image.LANCZOS)
    return img.crop(crop)

# Derive transform from person image and apply to ALL person-derived images
scale, crop = compute_transform(person_img)
print(f"Person transform: scale={scale:.3f}, crop={crop}")

person_r   = apply_transform(person_img,   scale, crop, "RGB")
agnostic_r = apply_transform(agnostic_img, scale, crop, "RGB")
mask_r     = apply_transform(mask_img,     scale, crop, "L")
pose_r     = apply_transform(pose_img,     scale, crop, "RGB")

# Garment has its own independent transform
cloth_scale, cloth_crop = compute_transform(cloth_img)
cloth_r = apply_transform(cloth_img, cloth_scale, cloth_crop, "RGB")

for name, img in [("person", person_r), ("agnostic", agnostic_r),
                   ("mask",    mask_r),   ("pose",     pose_r),
                   ("cloth",   cloth_r)]:
    print(f"  {name:10s} {img.size}  mode={img.mode}")

# Preview
fig, axes = plt.subplots(1, 5, figsize=(20, 6))
for ax, (name, img) in zip(axes, [("person", person_r), ("cloth", cloth_r),
                                   ("agnostic", agnostic_r), ("mask", mask_r),
                                   ("pose", pose_r)]):
    ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
    ax.set_title(f"{name}\n{img.size}")
    ax.axis('off')
plt.suptitle("Inputs to IDM-VTON (768×1024)", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Run IDM-VTON Inference — Our Stage 1-4 Inputs

The inference proceeds in two stages internally:
1. **Garment encoding** — `unet_encoder` processes `cloth_r` → garment feature maps
2. **Denoising** — `unet` iteratively denoises starting from random noise, guided by:
   - `agnostic_r` (cloth-removed person) as the inpainting base
   - `mask_r` (shirt region) marks where to inpaint
   - `pose_r` (skeleton) guides body structure
   - garment features (via IP-Adapter cross-attention) specify the target garment

> **T4 note:** With two SDXL UNets active, expect 5-10 minutes per image at `num_inference_steps=30`. If you hit OOM, change `cpu_offload = True` below.

In [ ]:
# If VRAM is tight, enable sequential CPU offload (much slower but won't OOM)
cpu_offload = False  # Set True if you hit CUDA out-of-memory errors
if cpu_offload:
    pipe.enable_sequential_cpu_offload()
    print("Sequential CPU offload enabled (slower but uses minimal VRAM)")

In [ ]:
import time

def run_idmvton(
    pipe,
    agnostic_img: Image.Image,
    mask_img: Image.Image,
    cloth_img: Image.Image,
    pose_img: Image.Image,
    prompt: str = "a photo of a person wearing a garment, photorealistic, high quality",
    neg_prompt: str = "monochrome, lowres, bad anatomy, worst quality, low quality, distorted",
    num_steps: int = 30,
    guidance_scale: float = 2.0,
    seed: int = 42,
    device: str = "cuda",
    dtype=torch.float16,
) -> Image.Image:
    """Run one IDM-VTON inference pass."""
    torch.cuda.empty_cache()
    t0 = time.time()

    with torch.no_grad():
        # Encode text prompts — SDXL uses two CLIP text encoders internally
        (prompt_embeds, neg_embeds,
         pooled_embeds, neg_pooled) = pipe.encode_prompt(
            prompt,
            num_images_per_prompt=1,
            do_classifier_free_guidance=True,
            negative_prompt=neg_prompt,
        )
        # Cloth prompt embedding (no CFG — used by garment encoder, not main UNet)
        (cloth_embeds, _, _, _) = pipe.encode_prompt(
            prompt,
            num_images_per_prompt=1,
            do_classifier_free_guidance=False,
        )

        output = pipe(
            prompt_embeds=prompt_embeds.to(device, dtype),
            negative_prompt_embeds=neg_embeds.to(device, dtype),
            pooled_prompt_embeds=pooled_embeds.to(device, dtype),
            negative_pooled_prompt_embeds=neg_pooled.to(device, dtype),
            num_inference_steps=num_steps,
            generator=torch.Generator(device).manual_seed(seed),
            strength=1.0,
            pose_img=pose_img,
            text_embeds_cloth=cloth_embeds.to(device, dtype),
            cloth=cloth_img,
            mask_image=mask_img,
            image=agnostic_img,
            height=TARGET_H,
            width=TARGET_W,
            ip_adapter_image=cloth_img,
            guidance_scale=guidance_scale,
        )

    elapsed = time.time() - t0
    print(f"Inference done in {elapsed:.1f}s")
    return output.images[0]

In [ ]:
# Run IDM-VTON on our Stage 1-4 outputs
print("Running IDM-VTON on our pipeline outputs (Stages 1-4)...")
print("  Input: test_person.jpg + clothing.jpg")
print("  Using: agnostic_person.png (Stage 2), erase_mask.png (Stage 2), pose_output.jpg (Stage 1)")

result_ours = run_idmvton(
    pipe,
    agnostic_img=agnostic_r,
    mask_img=mask_r,
    cloth_img=cloth_r,
    pose_img=pose_r,
    prompt="a person wearing a blue t-shirt, photorealistic, high quality",
    num_steps=30,
    seed=42,
)
print(f"Result size: {result_ours.size}")

In [ ]:
# Compare Stage 4 (Poisson blend) vs Stage 5 (IDM-VTON diffusion)
stage4_result = Image.open(f"{OUTPUT_DIR}/final_tryon.png").convert("RGB")

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, (title, img) in zip(axes, [
    ("Original person", person_r),
    ("Garment", cloth_r),
    ("Stage 4\n(Poisson blend)", stage4_result.resize((TARGET_W, TARGET_H), Image.LANCZOS)),
    ("Stage 5\n(IDM-VTON diffusion)", result_ours),
]):
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle("Virtual Try-On Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Save to Drive
result_ours.save(f"{OUTPUT_DIR}/stage5_result_ours.png")
print(f"Saved to {OUTPUT_DIR}/stage5_result_ours.png")

## 7. Run IDM-VTON on a VITON-HD Test Pair

Now we run the same pipeline on a VITON-HD test pair where:
- The agnostic image, mask, and pose are all **pre-computed from the dataset** (not our custom pipeline)
- This tells us how well IDM-VTON performs when given its "native" inputs
- Comparing against our custom inputs shows whether our Stages 1-3 produce equivalent quality

In [ ]:
# Prepare VITON-HD test pair inputs (already at 768×1024 natively)
# If some dataset inputs are missing, fall back to computing them from the person image

vton_agn_r  = vton_agnostic_img if vton_agnostic_img else vton_person_img
vton_mask_r = vton_mask_img     if vton_mask_img     else Image.new("L", (TARGET_W, TARGET_H), 128)
vton_pose_r = vton_pose_img     if vton_pose_img     else vton_person_img

# Ensure correct target size
def ensure_size(img, w=TARGET_W, h=TARGET_H, mode=None):
    if mode: img = img.convert(mode)
    if img.size != (w, h):
        img = img.resize((w, h), Image.LANCZOS)
    return img

vton_person_r = ensure_size(vton_person_img, mode="RGB")
vton_cloth_r  = ensure_size(vton_cloth_img,  mode="RGB")
vton_agn_r    = ensure_size(vton_agn_r,      mode="RGB")
vton_mask_r   = ensure_size(vton_mask_r,     mode="L")
vton_pose_r   = ensure_size(vton_pose_r,     mode="RGB")

print("VITON-HD test pair prepared.")
for name, img in [("person", vton_person_r), ("cloth", vton_cloth_r),
                   ("agnostic", vton_agn_r), ("mask", vton_mask_r)]:
    print(f"  {name:10s} {img.size}")

In [ ]:
print("Running IDM-VTON on VITON-HD test pair...")
result_vton = run_idmvton(
    pipe,
    agnostic_img=vton_agn_r,
    mask_img=vton_mask_r,
    cloth_img=vton_cloth_r,
    pose_img=vton_pose_r,
    prompt="a person wearing a garment, photorealistic, high quality",
    num_steps=30,
    seed=0,
)

# Show result
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, (title, img) in zip(axes, [
    ("VITON-HD person", vton_person_r),
    ("VITON-HD garment", vton_cloth_r),
    ("Agnostic person", vton_agn_r),
    ("IDM-VTON result", result_vton),
]):
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle("IDM-VTON on VITON-HD Test Pair", fontsize=14)
plt.tight_layout()
plt.show()

result_vton.save(f"{OUTPUT_DIR}/stage5_result_vitonhd.png")
print(f"Saved to {OUTPUT_DIR}/stage5_result_vitonhd.png")

## 8. Save Results + Model Info to Drive

In [ ]:
# Save side-by-side comparison of full pipeline (Stages 1-5)
stage4_r = Image.open(f"{OUTPUT_DIR}/final_tryon.png").convert("RGB").resize(
    (TARGET_W, TARGET_H), Image.LANCZOS
)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, (title, img) in zip(axes, [
    ("Person", person_r),
    ("Stage 4: Poisson blend", stage4_r),
    ("Stage 5: IDM-VTON", result_ours),
]):
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis('off')
plt.suptitle("Virtual Try-On: Classical vs Neural", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/stage5_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Comparison saved to {OUTPUT_DIR}/stage5_comparison.png")

In [ ]:
import json
from datetime import datetime

# Save model info / run config to Drive checkpoints folder
# (Actual weights are NOT saved to Drive — too large. Re-download each session.)
run_info = {
    "stage": 5,
    "date": datetime.now().isoformat(),
    "model": {
        "repo": "yisol/IDM-VTON",
        "base": "stabilityai/stable-diffusion-xl-base-1.0",
        "vae": "madebyollin/sdxl-vae-fp16-fix",
        "scheduler": "DDIMScheduler",
    },
    "inference": {
        "resolution": f"{TARGET_W}x{TARGET_H}",
        "num_inference_steps": 30,
        "guidance_scale": 2.0,
        "dtype": "float16",
        "memory_opts": ["attention_slicing", "xformers"],
    },
    "inputs": {
        "agnostic": "output_images/agnostic_person.png  (Stage 2)",
        "mask":     "output_images/erase_mask.png        (Stage 2)",
        "pose":     "output_images/pose_output.jpg       (Stage 1)",
        "cloth":    "input_images/clothing.jpg",
    },
    "outputs": {
        "ours":    "output_images/stage5_result_ours.png",
        "vitonhd": "output_images/stage5_result_vitonhd.png",
        "compare": "output_images/stage5_comparison.png",
    },
    "notes": "Fresh Colab T4 runtime. huggingface_hub pinned to avoid system conflict.",
}

info_path = f"{CKPT_DIR}/stage5_run_info.json"
with open(info_path, "w") as f:
    json.dump(run_info, f, indent=2)

print("Run info saved to Drive:")
print(json.dumps(run_info, indent=2))

## Summary

**What Stage 5 achieved:**
- Downloaded VITON-HD dataset (fresh each session, not saved to Drive)
- Loaded IDM-VTON pre-trained weights with fp16 + xformers for T4 compatibility
- Ran neural try-on inference using our Stage 1-4 outputs as direct inputs — no reprocessing needed
- Compared classical (Poisson blend) vs neural (diffusion) compositing

**Why it works:** Our Stages 1-3 produce the exact intermediate representations IDM-VTON expects:
- Stage 2 agnostic person → IDM-VTON's cloth-agnostic input
- Stage 2 erase mask → IDM-VTON's inpainting mask
- Stage 1 skeleton → IDM-VTON's pose conditioning

**What's next — Stage 6:**
Fine-tune IDM-VTON on VITON-HD training split using mixed precision (fp16) on T4. This adapts the pre-trained weights to improve performance on the specific clothing styles in the dataset.

| Stage | What |
|---|---|
| ~~1~~ | Pose estimation ✅ |
| ~~2~~ | Body segmentation ✅ |
| ~~3~~ | Clothing warping ✅ |
| ~~4~~ | Poisson blending ✅ |
| ~~5~~ | IDM-VTON pre-trained inference ✅ |
| **6** | Fine-tune on VITON-HD (T4, mixed precision) |
| 7 | FastAPI endpoint |
| 8 | Deploy on RunPod/Modal |